In [ ]:
import pandas as pd
import re
from datasets import load_dataset, Dataset, DatasetDict

In [ ]:
print("Loading tatsu-lab/alpaca dataset.....")
dataset = load_dataset("tatsu-lab/alpaca", split="train")

Loading tatsu-lab/alpaca dataset.....


README.md:   0%|          | 0.00/7.47k [00:00<?, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…):   0%|          | 0.00/24.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

In [ ]:
df = dataset.to_pandas()

In [ ]:
code_keywords = [
    'python', 'javascript', 'html', 'css', 'c++', 'java',
    'function', 'algorithm', 'code', 'script', 'debug', 'sql'
]
pattern = '|'.join(code_keywords)

print("Filtering for code Generation domain ....")

df_code = df[
    df['instruction'].str.contains(pattern, case=False, na=False)|
    df['input'].str.contains(pattern, case=False, na=False)
    ].copy()


Filtering for code Generation domain ....


In [ ]:
print(f"Initial domain sample extracted : {len(df_code)}")

Initial domain sample extracted : 45521


In [ ]:
df_clean = df_code.drop_duplicates(subset=['instruction', 'input', 'output'])

In [ ]:
def clean_text(text):
  if not isinstance(text, str): return ""

  text = text.strip()
  text = re.sub(r'\n{3,}', '\n\n', text)
  return text


In [ ]:
df_clean.loc[:, 'instruction'] = df_clean['instruction'].apply(clean_text)
df_clean.loc[:, 'input'] = df_clean['input'].apply(clean_text)
df_clean.loc[:, 'output'] = df_clean['output'].apply(clean_text)

In [ ]:
df_clean = df_clean[df_clean['output'].str.len() > 30]

target_size = min(800, len(df_clean))
df_final = df_clean.sample(n=target_size, random_state=42).reset_index(drop=True)

print(f"sample after cleaning and downsampling: {len(df_final)}")

sample after cleaning and downsampling: 800


In [ ]:
final_dataset = Dataset.from_pandas(df_final)

In [ ]:
train_test = final_dataset.train_test_split(test_size=0.2, seed=42)
val_test = train_test['test'].train_test_split(test_size=0.5, seed=42)

dataset_dict = DatasetDict({
    'train':      train_test["train"],
    'validation': val_test["train"],
    'test':       val_test["test"]
})

print("datasett splits complete ")
print(f"train: {len(dataset_dict['train'])}")

datasett splits complete 
train: 640


In [ ]:
print(f"Validation: {len(dataset_dict['validation'])}")
print(f"test: {len(dataset_dict['test'])}")

Validation: 80
test: 80


In [ ]:
import huggingface_hub
huggingface_hub.login()

In [ ]:
hf_repo_name = "FathyElghoneimy/alpaca-code-generation-curated"

try:
  print(f"pushinf dataset to HuggingFace Hub at {hf_repo_name}")
  dataset_dict.push_to_hub(hf_repo_name)
  print("done")
except Exception as e:
  print("Failed to push to Hugging Face. Did you run `huggingface-cli login`?")
  print(f"Error state: {e}")

pushinf dataset to HuggingFace Hub at FathyElghoneimy/alpaca-code-generation-curated


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 47.9kB / 47.9kB            

README.md:   0%|          | 0.00/595 [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


done


In [ ]:
from google.colab import userdata
from huggingface_hub import login

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("Successfully logged into Hugging Face Hub!")
except Exception as e:
    print(f"Login failed: {e}")
    print("\nFIX: Go to the Secrets tab (key icon), add 'HF_TOKEN', and enable 'Notebook access'.")

Successfully logged into Hugging Face Hub!


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_config
from trl import SFTTrainer, SFTConfig

In [ ]:
dataset_id = "FathyElghoneimy/alpaca-code-generation-curated"
dataset = load_dataset(dataset_id)
print(f"dataset loaded from: {dataset_id}")

dataset loaded from: FathyElghoneimy/alpaca-code-generation-curated


In [ ]:
def format_instruction(example):
  if example.get("input", ""):
    return f"### Instruction:\n{example['instruction']}\n\n### Input:\n{example['input']}\n\n### Response:\n{example['output']}"
  else:
    return f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}"

In [ ]:
model_id = "NousResearch/Meta-Llama-3-8B"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blo

generation_config.json:   0%|          | 0.00/177 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

In [ ]:
peft_config= LoraConfig(
    r= 16,
    lora_alpha= 32,
    lora_dropout=0.05,
    bias= "none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"] #target attention layers
)


In [ ]:
training_args= SFTConfig(
    output_dir= "./lora_adapters",
    per_device_train_batch_size= 2,
    gradient_accumulation_steps= 4,
    learning_rate= 2e-4,
    optim= "paged_adamw_8bit",
    num_train_epochs=1,
    logging_steps=10,
    fp16=False
)

In [ ]:
trainer = SFTTrainer(
    model= model,
    train_dataset= dataset["train"],
    eval_dataset= dataset["validation"],
    peft_config= peft_config,
    args= training_args,
    formatting_func= format_instruction
)

Applying formatting function to train dataset:   0%|          | 0/640 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/640 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/640 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/640 [00:00<?, ? examples/s]

Applying formatting function to eval dataset:   0%|          | 0/80 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/80 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/80 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/80 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.


Step,Training Loss
10,1.772566
20,1.574487
30,1.468320
40,1.371829
50,1.295870
60,1.286116
70,1.264048
80,1.393641


TrainOutput(global_step=80, training_loss=1.4283596515655517, metrics={'train_runtime': 2256.9096, 'train_samples_per_second': 0.284, 'train_steps_per_second': 0.035, 'total_flos': 3433784746721280.0, 'train_loss': 1.4283596515655517, 'epoch': 1.0})

In [ ]:
trainer.model.save_pretrained("my-custom-lora-adapter")
print("✅ LoRA adapters saved locally to Colab!")

✅ LoRA adapters saved locally to Colab!
